Step 4- Adding severity labels based on clinical outcome flags.

In [1]:
import pandas as pd

SRC = "../data/processed/sample_1k.csv"
OUT = "../data/processed/sample_1k_with_severity.csv"

# Columns expected to exist in SRC
FLAG_COLS = ["DIED", "L_THREAT", "HOSPITAL", "DISABLE", "ER_VISIT"]
ALLOWED = {"mild", "moderate", "severe"}

# --- Load ---
df = pd.read_csv(SRC, low_memory=False)

# Validate required columns
missing = [c for c in FLAG_COLS if c not in df.columns]
if missing:
    raise ValueError(f"Missing columns in {SRC}: {missing}")

# --- Normalize flags to 0/1 robustly ---
# Any variant like "Y", "Yes", "y", " y " -> 1; else 0
for col in FLAG_COLS:
    df[col] = (
        df[col]
        .astype(str)
        .str.strip()
        .str.upper()
        .str.contains(r"\bY(ES)?\b", na=False)
        .astype(int)
    )

# --- Map severity (severe > moderate > mild) ---
def map_severity(row) -> str:
    # Severe: death, life-threatening, or permanent disability
    if row["DIED"] == 1 or row["L_THREAT"] == 1 or row["DISABLE"] == 1:
        return "severe"
    # Moderate: hospitalization or ER visit
    if row["HOSPITAL"] == 1 or row["ER_VISIT"] == 1:
        return "moderate"
    # Otherwise, mild
    return "mild"

df["severity_label"] = df.apply(map_severity, axis=1)

# --- Sanity: force canonical values & validate ---
df["severity_label"] = (
    df["severity_label"].astype(str).str.strip().str.lower()
)

unexpected = set(df["severity_label"].unique()) - ALLOWED
if unexpected:
    raise ValueError(f"Found unexpected severity values: {unexpected}")

# Optional: lock to categorical
df["severity_label"] = pd.Categorical(
    df["severity_label"], categories=["mild", "moderate", "severe"], ordered=True
).astype(str)

# --- Save & report ---
df.to_csv(OUT, index=False)
print(f"✅ Saved with severity labels to {OUT}")
print("\nSeverity distribution:")
print(df["severity_label"].value_counts())


✅ Saved with severity labels to ../data/processed/sample_1k_with_severity.csv

Severity distribution:
severity_label
mild        884
moderate     73
severe       43
Name: count, dtype: int64


C:\Users\Mochitha vijayan\AppData\Local\Temp\ipykernel_2732\3611934526.py:26: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  .str.contains(r"\bY(ES)?\b", na=False)
C:\Users\Mochitha vijayan\AppData\Local\Temp\ipykernel_2732\3611934526.py:26: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  .str.contains(r"\bY(ES)?\b", na=False)
C:\Users\Mochitha vijayan\AppData\Local\Temp\ipykernel_2732\3611934526.py:26: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  .str.contains(r"\bY(ES)?\b", na=False)
C:\Users\Mochitha vijayan\AppData\Local\Temp\ipykernel_2732\3611934526.py:26: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  .str.contains(r"\bY(ES)?\b", na